In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

In [ ]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [ ]:
import os, mlflow
print("ENV MLFLOW_TRACKING_URI =", os.getenv("MLFLOW_TRACKING_URI"))
print("mlflow.get_tracking_uri() =", mlflow.get_tracking_uri())

## 1. Setup

SegFormer (nvidia/mit-b2) baseline — Issue [#10](https://github.com/data-mermaid/mermaid-segmentation/issues/10)

In [ ]:
import copy

import albumentations as A
import mermaidseg.datasets.dataset
import torch
from torch.utils.data import DataLoader

from mermaidseg.io import setup_config, get_parser, update_config_with_args
from mermaidseg.logger import Logger
from mermaidseg.model.eval import EvaluatorSemanticSegmentation
from mermaidseg.model.meta import MetaModel
from mermaidseg.model.train import train_model

In [ ]:
device_count = torch.cuda.device_count()
for i in range(device_count):
    print(f"CUDA Device {i}: {torch.cuda.get_device_name(i)}")

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

seed = 42
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)

## 2. Config

In [ ]:
cfg = setup_config(
    config_path="../configs/segformer-mit-b2.yaml",
    config_base_path="../configs/base_mermaid.yaml",
)

args_input = "--run-name=segformer_mit_b2_baseline --log-epochs=1"
args_input = args_input.split(" ")

parser = get_parser()
args = parser.parse_args(args_input)

cfg = update_config_with_args(cfg, args)
cfg_logger = copy.deepcopy(cfg)

In [ ]:
os.environ.pop("MLFLOW_TRACKING_URI", None)
mlflow.set_tracking_uri("./segmentation")
cfg_logger.logger.uri = "./segmentation"
cfg_logger.logger.system_metrics = True
cfg_logger.logger.system_metrics_sampling_interval = 2

print("AFTER override:", mlflow.get_tracking_uri())

## 3. Data

In [ ]:
transforms = {}
for split in cfg.augmentation:
    transforms[split] = A.Compose(
        [
            getattr(A, transform_name)(**transform_params)
            for transform_name, transform_params in cfg.augmentation[split].items()
        ]
    )

In [ ]:
dataset_name = cfg.data.pop("name", None)
batch_size   = cfg.data.pop("batch_size", 4)

DatasetClass = getattr(mermaidseg.datasets.dataset, dataset_name)

# Load once (no transform) to determine split indices
index_dataset = DatasetClass(transform=None, **cfg.data)
total_size = len(index_dataset)
train_size = int(0.7 * total_size)
val_size   = int(0.15 * total_size)
test_size  = total_size - train_size - val_size

generator = torch.Generator().manual_seed(42)
train_indices, val_indices, test_indices = [
    subset.indices
    for subset in random_split(index_dataset, [train_size, val_size, test_size], generator=generator)
]

# Create per-split datasets with the correct transforms
train_dataset = torch.utils.data.Subset(
    DatasetClass(transform=transforms["train"], **cfg.data), train_indices
)
val_dataset = torch.utils.data.Subset(
    DatasetClass(transform=transforms["val"], **cfg.data), val_indices
)
test_dataset = torch.utils.data.Subset(
    DatasetClass(transform=transforms["test"], **cfg.data), test_indices
)

# Reference dataset for collate_fn and metadata
full_dataset = DatasetClass(transform=transforms["train"], **cfg.data)

print(f"Total: {total_size} → Train: {train_size}, Val: {val_size}, Test: {test_size}")

In [ ]:
generator = torch.Generator().manual_seed(42)
train_dataset, val_dataset, test_dataset = random_split(
    full_dataset, [train_size, val_size, test_size], generator=generator
)

In [ ]:
# Subset for smoke-testing — comment out for a full training run
train_dataset = torch.utils.data.Subset(train_dataset, range(10))
val_dataset   = torch.utils.data.Subset(val_dataset,   range(10))
test_dataset  = torch.utils.data.Subset(test_dataset,  range(10))

In [ ]:
import time

def benchmark_num_workers(dataset, collate_fn, batch_size: int, candidates=(0, 1, 2, 4), n_batches: int = 5):
    """
    Times how long it takes to iterate over `n_batches` batches for each candidate
    num_workers value and returns the fastest option.

    S3-backed datasets are I/O-bound, so the optimal value is usually higher than 0
    but can plateau or degrade if workers compete for bandwidth or memory.
    """
    results = {}
    for nw in candidates:
        loader = DataLoader(dataset, batch_size=batch_size, num_workers=nw,
                            collate_fn=collate_fn, shuffle=True, drop_last=True)
        t0 = time.perf_counter()
        for i, _ in enumerate(loader):
            if i + 1 >= n_batches:
                break
        results[nw] = time.perf_counter() - t0
        print(f"  num_workers={nw}: {results[nw]:.2f}s for {n_batches} batches")

    best = min(results, key=results.get)
    print(f"\nFastest: num_workers={best}")
    return best

# Heuristic default — avoids launching a full benchmark every run.
# Uncomment the benchmark call when tuning a new machine or dataset.
num_workers = min(4, os.cpu_count() or 1)
print(f"Default num_workers={num_workers} (cpu_count={os.cpu_count()})")

# num_workers = benchmark_num_workers(
#     train_dataset, full_dataset.collate_fn, batch_size, candidates=(0, 1, 2, 4)
# )

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    drop_last=True,
    collate_fn=full_dataset.collate_fn,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    drop_last=True,
    collate_fn=full_dataset.collate_fn,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    drop_last=True,
    collate_fn=full_dataset.collate_fn,
)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}")

In [ ]:
print(f"num_classes: {full_dataset.num_classes}")

In [ ]:
## 4. Model (SegFormer)

meta_model = MetaModel(
    run_name=cfg.run_name,
    num_classes=full_dataset.num_classes,
    device=device,
    model_kwargs=cfg.model,
    training_kwargs=cfg.training,
)

evaluator = EvaluatorSemanticSegmentation(
    num_classes=full_dataset.num_classes,
    device=device,
)

## 5. Logger

In [ ]:
os.environ.pop("MLFLOW_TRACKING_URI", None)

cfg_logger.logger.experiment_name = "segformer_baseline"
cfg_logger.logger.uri = "./segmentation"

logger = Logger(
    config=cfg_logger,
    meta_model=meta_model,
    log_epochs=cfg.logger.log_epochs,
    log_checkpoint=2,
    checkpoint_dir=".",
    enable_mlflow=True,
    enable_wandb=False,
    save_local_checkpoints=True,
    id2label=full_dataset.id2label,
)

## 6. Training

In [ ]:
train_model(meta_model, evaluator, train_loader, val_loader, logger=logger)

In [ ]:
print(mlflow.get_tracking_uri())